# SQL Analysis — Drone Fleet Power Consumption

**Dataset:** CMU DJI Matrice 100, 188 flights, 177,464 telemetry rows  
**Database:** `flights.db` (SQLite3, loaded from `real_data/flights_ready_for_eda.csv`)  
**Scope:** 5 research-quality queries, one per Research Question (RQ-A through RQ-E)

Techniques used across queries:
- CTE (Common Table Expression)
- GROUP BY with aggregation
- Self-JOIN with z-score
- Window functions: `RANK() OVER PARTITION BY`, `PERCENT_RANK() OVER PARTITION BY`
- `CASE WHEN` categorical binning

In [20]:
import sqlite3
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

DB_PATH = os.path.join(os.getcwd(), 'flights.db')
VIZ_DIR = os.getcwd()
BASE_DIR = os.path.dirname(os.path.abspath('02_sql_analysis.ipynb'))
_RESULTS_DIR = os.path.join(BASE_DIR, '..', 'results')
os.makedirs(_RESULTS_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)
conn = sqlite3.connect(DB_PATH)
print(f'Connected: {DB_PATH}')
print(f'Rows in flights table: {pd.read_sql("SELECT COUNT(*) AS n FROM flights", conn).iloc[0,0]:,}')

Connected: c:\Users\thien\code\AIMY\drone\sql\flights.db
Rows in flights table: 177,464


---
## RQ-A: Does headwind correlate with Power consumption?

**Hypothesis (Simpson's Paradox):** Pooled correlation is near zero, but per-regime direction reverses.  
**Technique:** CTE + GROUP BY with manual Pearson formula (SQLite omits `CORR()`)

In [ ]:
qA2_1 = """
WITH wind_bins AS (
    SELECT
        regime,
        CASE
            WHEN headwind < -3 THEN 'Strong Tailwind'
            WHEN headwind >= -3 AND headwind < 0 THEN 'Mild Tailwind'
            WHEN headwind >= 0 AND headwind < 3 THEN 'Mild Headwind'
            ELSE 'Strong Headwind'
        END AS wind_bin,
        Power
    FROM flights
),
agg AS (
    SELECT
        regime,
        wind_bin,
        COUNT(*) AS n_rows,
        ROUND(AVG(Power), 2) AS avg_power_W,
        ROUND(MIN(Power), 2) AS min_power_W,
        ROUND(MAX(Power), 2) AS max_power_W
    FROM wind_bins
    GROUP BY regime, wind_bin
)
SELECT *
FROM agg
ORDER BY regime,
         CASE wind_bin
            WHEN 'Strong Tailwind' THEN 1
            WHEN 'Mild Tailwind'   THEN 2
            WHEN 'Mild Headwind'   THEN 3
            WHEN 'Strong Headwind' THEN 4
         END;
"""
df_qA2_1 = pd.read_sql(qA2_1, conn)
display(df_qA2_1)

,regime,wind_bin,n_rows,avg_power_W,min_power_W,max_power_W
0,cruise,Strong Tailwind,29944,499.27,124.77,954.70
1,cruise,Mild Tailwind,14994,524.44,212.09,934.21
2,cruise,Mild Headwind,17050,525.06,227.26,925.72
3,cruise,Strong Headwind,33883,519.70,106.48,946.70
4,landing,Strong Tailwind,3245,510.14,24.50,870.38
5,landing,Mild Tailwind,21157,494.45,7.32,878.65
6,landing,Mild Headwind,17425,499.01,3.34,1004.22
7,landing,Strong Headwind,8866,475.58,128.82,909.16
8,takeoff,Strong Tailwind,3900,619.34,28.96,925.06
9,takeoff,Mild Tailwind,12005,604.15,43.03,951.79


In [21]:
qA2_2 = """
WITH stats AS (
  SELECT regime,
         AVG(headwind)         AS mh,
         AVG(Power)            AS mp,
         AVG(headwind*headwind) AS mh2,
         AVG(Power*Power)       AS mp2,
         AVG(headwind*Power)    AS mhp
  FROM flights
  GROUP BY regime
)
SELECT regime,
       ROUND((mhp - mh*mp) / (SQRT(mh2 - mh*mh) * SQRT(mp2 - mp*mp)), 4)
         AS corr_headwind_power
FROM stats;
"""
df_qA2_2 = pd.read_sql(qA2_2, conn)
display(df_qA2_2)

fig, ax = plt.subplots(figsize=(6, 4))
colors = {'cruise': '#1976D2', 'landing': '#388E3C', 'takeoff': '#E91E63'}
bars = ax.barh(df_qA2_2['regime'], df_qA2_2['corr_headwind_power'],
               color=[colors[r] for r in df_qA2_2['regime']])
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, df_qA2_2['corr_headwind_power']):
    ax.text(val + (0.003 if val >= 0 else -0.003), bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)
ax.set_xlabel('Pearson r (headwind vs. Power)')
ax.set_title('RQ-A: Per-Regime Headwind–Power Correlation\n(pooled r \u2248 0 masks opposing regime directions)')
ax.set_xlim(-0.20, 0.22)
plt.tight_layout()
out = os.path.join(_RESULTS_DIR, 'sql_qA2_corr.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

,regime,corr_headwind_power
0,cruise,0.1216
1,landing,-0.1041
2,takeoff,-0.0649


Saved: c:\Users\thien\code\AIMY\drone\sql\..\results\sql_qA2_corr.png


---
## RQ-B: Which flights consume the most power per regime?

**Purpose:** Identify the top-3 highest-average-power flights within each regime to understand what flight configurations drive energy demand.  
**Technique:** CTE + `RANK() OVER (PARTITION BY regime ORDER BY avg_power DESC)` — window function

In [15]:
qB2_1 = """
WITH flight_regime_avg AS (
    SELECT
        flight,
        regime,
        AVG(Power) AS flight_avg_power
    FROM flights
    GROUP BY flight, regime
),
regime_baseline AS (
    SELECT
        regime,
        AVG(Power) AS regime_avg_power
    FROM flights
    GROUP BY regime
),
scored AS (
    SELECT
        f.flight,
        f.regime,
        ROUND(f.flight_avg_power, 2) AS flight_avg_power_W,
        ROUND(r.regime_avg_power, 2) AS regime_avg_power_W,
        ROUND(f.flight_avg_power - r.regime_avg_power, 2) AS power_gap_W,
        ROUND(100.0 * (f.flight_avg_power - r.regime_avg_power) / r.regime_avg_power, 2) AS pct_above_regime,
        RANK() OVER (
            PARTITION BY f.regime
            ORDER BY (f.flight_avg_power - r.regime_avg_power) DESC
        ) AS deviation_rank
    FROM flight_regime_avg f
    JOIN regime_baseline r
      ON f.regime = r.regime
)
SELECT
    regime,
    flight,
    flight_avg_power_W,
    regime_avg_power_W,
    power_gap_W,
    pct_above_regime,
    deviation_rank
FROM scored
WHERE deviation_rank <= 5
ORDER BY regime, deviation_rank;
"""
df_qB2_1 = pd.read_sql(qB2_1, conn)
display(df_qB2_1)

,regime,flight,flight_avg_power_W,regime_avg_power_W,power_gap_W,pct_above_regime,deviation_rank
0,cruise,129,610.79,515.01,95.78,18.60,1
1,cruise,138,607.84,515.01,92.83,18.02,2
2,cruise,130,593.71,515.01,78.70,15.28,3
3,cruise,232,591.60,515.01,76.59,14.87,4
4,cruise,155,589.50,515.01,74.49,14.46,5
5,landing,105,576.74,493.72,83.02,16.81,1
6,landing,99,565.73,493.72,72.01,14.58,2
7,landing,107,564.77,493.72,71.05,14.39,3
8,landing,106,564.74,493.72,71.03,14.39,4
9,landing,140,560.94,493.72,67.23,13.62,5


In [22]:
qB2_2 = """
WITH flight_regime_avg AS (
  SELECT flight, regime, AVG(Power) AS avg_power
  FROM flights
  GROUP BY flight, regime
),
ranked AS (
  SELECT regime, flight, avg_power,
         RANK() OVER (PARTITION BY regime ORDER BY avg_power DESC) AS power_rank
  FROM flight_regime_avg
)
SELECT regime, flight, ROUND(avg_power, 2) AS avg_power_W, power_rank
FROM ranked
WHERE power_rank <= 3
ORDER BY regime, power_rank;
"""
df_qB2_2 = pd.read_sql(qB2_2, conn)
display(df_qB2_2)

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'cruise': '#1976D2', 'landing': '#388E3C', 'takeoff': '#E91E63'}
y_pos = np.arange(len(df_qB2_2))
bar_colors = [colors[r] for r in df_qB2_2['regime']]
bars = ax.barh(y_pos, df_qB2_2['avg_power_W'], color=bar_colors)
labels = [f"Flight {int(r['flight'])} ({r['regime']}, Rank {int(r['power_rank'])})" 
          for _, r in df_qB2_2.iterrows()]
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
for bar, val in zip(bars, df_qB2_2['avg_power_W']):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, f'{val:.1f} W',
            va='center', fontsize=8)
ax.set_xlabel('Average Power (W)')
ax.set_title('RQ-B: Top-3 Highest-Power Flights per Regime\n(RANK window function)')
ax.set_xlim(0, 800)
plt.tight_layout()
out = os.path.join(_RESULTS_DIR, 'sql_qB2_top_flights.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

,regime,flight,avg_power_W,power_rank
0,cruise,129,610.79,1
1,cruise,138,607.84,2
2,cruise,130,593.71,3
3,landing,105,576.74,1
4,landing,99,565.73,2
5,landing,107,564.77,3
6,takeoff,126,713.46,1
7,takeoff,231,695.94,2
8,takeoff,130,693.11,3


Saved: c:\Users\thien\code\AIMY\drone\sql\..\results\sql_qB2_top_flights.png


---
## RQ-C: Where do anomalous sensor readings concentrate?

**Purpose:** Flag individual telemetry rows with acceleration z-score > 2 (outliers within their flight). High anomaly density = high robustness risk zones — sensors are already noisy before any synthetic perturbation.  
**Technique:** Self-JOIN between row-level data and per-flight statistics, then z-score filter

In [23]:
qC2 = """
WITH flight_stats AS (
  SELECT flight,
         AVG(accel_mag)  AS mean_accel,
         SQRT(AVG(accel_mag*accel_mag) - AVG(accel_mag)*AVG(accel_mag)) AS std_accel
  FROM flights
  GROUP BY flight
)
SELECT f.flight, f.regime, f.accel_mag, f.Power,
       ROUND((f.accel_mag - fs.mean_accel) / fs.std_accel, 2) AS accel_zscore
FROM flights f
JOIN flight_stats fs ON f.flight = fs.flight
WHERE ABS((f.accel_mag - fs.mean_accel) / fs.std_accel) > 2
ORDER BY accel_zscore DESC
LIMIT 20;
"""
df_qC2 = pd.read_sql(qC2, conn)
display(df_qC2)

# Count anomalous rows per regime (full table, not just LIMIT 20)
qC2_count = """
WITH flight_stats AS (
  SELECT flight,
         AVG(accel_mag) AS mean_accel,
         SQRT(AVG(accel_mag*accel_mag) - AVG(accel_mag)*AVG(accel_mag)) AS std_accel
  FROM flights GROUP BY flight
)
SELECT f.regime, COUNT(*) AS anomalous_rows
FROM flights f
JOIN flight_stats fs ON f.flight = fs.flight
WHERE ABS((f.accel_mag - fs.mean_accel) / fs.std_accel) > 2
GROUP BY f.regime ORDER BY f.regime;
"""
df_qC2_cnt = pd.read_sql(qC2_count, conn)
display(df_qC2_cnt)

fig, ax = plt.subplots(figsize=(6, 4))
colors = {'cruise': '#1976D2', 'landing': '#388E3C', 'takeoff': '#E91E63'}
ax.bar(df_qC2_cnt['regime'], df_qC2_cnt['anomalous_rows'],
       color=[colors[r] for r in df_qC2_cnt['regime']])
for i, (_, row) in enumerate(df_qC2_cnt.iterrows()):
    ax.text(i, row['anomalous_rows'] + 30, f"{int(row['anomalous_rows']):,}",
            ha='center', fontsize=10)
ax.set_ylabel('Anomalous Rows (|z| > 2)')
ax.set_title('RQ-C: Anomalous Acceleration Readings per Regime\n(self-JOIN + z-score filter)')
plt.tight_layout()
out = os.path.join(_RESULTS_DIR, 'sql_qC2_anomaly.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

,flight,regime,accel_mag,Power,accel_zscore
0,148,landing,20.380602,5.736462,21.40
1,2,landing,23.639402,12.536397,18.31
2,1,landing,18.452596,11.998961,16.95
3,131,landing,18.785738,232.012290,16.93
4,126,landing,19.288545,258.262455,16.69
5,8,landing,17.187267,8.619249,16.62
6,109,landing,17.463851,588.885183,16.52
7,231,landing,18.126208,228.938581,15.41
8,118,landing,17.101238,438.236438,14.90
9,110,landing,16.272918,209.046402,14.83


,regime,anomalous_rows
0,cruise,3141
1,landing,3384
2,takeoff,3813


Saved: c:\Users\thien\code\AIMY\drone\sql\..\results\sql_qC2_anomaly.png


---
## RQ-D: Does headwind direction affect Power differently across regimes?

**Purpose:** Validate SHAP's headwind direction annotation. Physics predicts headwind increases power in cruise (drag), but takeoff relies on vertical thrust so horizontal wind matters less.  
**Technique:** CTE + `CASE WHEN` categorical binning + GROUP BY (regime × wind direction)

In [20]:
qD2_1 = """
WITH wind_class AS (
    SELECT
        regime,
        CASE
            WHEN headwind <= -5 THEN 'Strong Tailwind'
            WHEN headwind > -5 AND headwind < 0 THEN 'Light Tailwind'
            WHEN headwind >= 0 AND headwind < 5 THEN 'Light Headwind'
            ELSE 'Strong Headwind'
        END AS wind_class,
        Power
    FROM flights
),
agg AS (
    SELECT
        regime,
        wind_class,
        COUNT(*) AS n_rows,
        ROUND(AVG(Power), 2) AS avg_power_W,
        SQRT(AVG(Power * Power) - AVG(Power) * AVG(Power)) AS power_std_W
    FROM wind_class
    GROUP BY regime, wind_class
)
SELECT
    regime,
    wind_class,
    n_rows,
    avg_power_W,
    power_std_W
FROM agg
ORDER BY regime,
         CASE wind_class
            WHEN 'Strong Tailwind' THEN 1
            WHEN 'Light Tailwind'  THEN 2
            WHEN 'Light Headwind'  THEN 3
            WHEN 'Strong Headwind' THEN 4
         END;
"""
df_qD2_1 = pd.read_sql(qD2_1, conn)
display(df_qD2_1)

,regime,wind_class,n_rows,avg_power_W,power_std_W
0,cruise,Strong Tailwind,20729,495.11,72.377555
1,cruise,Light Tailwind,24209,518.42,79.888867
2,cruise,Light Headwind,28375,521.04,72.757212
3,cruise,Strong Headwind,22558,522.07,88.802751
4,landing,Strong Tailwind,463,488.14,116.105251
5,landing,Light Tailwind,23939,496.70,91.630395
6,landing,Light Headwind,22208,499.32,76.091274
7,landing,Strong Headwind,4083,446.41,96.620008
8,takeoff,Strong Tailwind,305,581.23,83.858892
9,takeoff,Light Tailwind,15600,608.40,94.010815


In [24]:
qD2_2 = """
WITH binned AS (
  SELECT regime,
         CASE WHEN headwind < 0 THEN 'tailwind' ELSE 'headwind' END AS wind_dir,
         Power
  FROM flights
)
SELECT regime, wind_dir, COUNT(*) AS n,
       ROUND(AVG(Power), 2) AS avg_power_W
FROM binned
GROUP BY regime, wind_dir
ORDER BY regime, wind_dir;
"""
df_qD2_2 = pd.read_sql(qD2_2, conn)
display(df_qD2_2)

pivot = df_qD2_2.pivot(index='regime', columns='wind_dir', values='avg_power_W')
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(pivot))
w = 0.35
b1 = ax.bar(x - w/2, pivot['headwind'], w, label='Headwind', color='#E53935')
b2 = ax.bar(x + w/2, pivot['tailwind'],  w, label='Tailwind',  color='#43A047')
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('Average Power (W)')
ax.set_title('RQ-D: Avg Power by Wind Direction x Regime\n(validates SHAP headwind direction annotation)')
ax.legend()
ax.set_ylim(0, 680)
plt.tight_layout()
out = os.path.join(_RESULTS_DIR, 'sql_qD2_headwind_regime.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

,regime,wind_dir,n,avg_power_W
0,cruise,headwind,50933,521.49
1,cruise,tailwind,44938,507.67
2,landing,headwind,26291,491.11
3,landing,tailwind,24402,496.53
4,takeoff,headwind,14995,592.63
5,takeoff,tailwind,15905,607.88


Saved: c:\Users\thien\code\AIMY\drone\sql\..\results\sql_qD2_headwind_regime.png


---
## RQ-E: Which flights lie in the high-uncertainty zone?

**Purpose:** Identify the top-10% most variable flights per regime using `PERCENT_RANK`. These are the flights where both prediction errors AND SHAP instability are expected to be highest (supporting the r=0.919 coupling finding).  
**Technique:** CTE + `PERCENT_RANK() OVER (PARTITION BY regime ORDER BY std_accel)`

In [25]:
qE2 = """
WITH flight_var AS (
  SELECT flight, regime,
         SQRT(AVG(accel_mag*accel_mag) - AVG(accel_mag)*AVG(accel_mag)) AS std_accel
  FROM flights
  GROUP BY flight, regime
),
ranked AS (
  SELECT flight, regime, std_accel,
         PERCENT_RANK() OVER (PARTITION BY regime ORDER BY std_accel) AS pct_rank
  FROM flight_var
)
SELECT flight, regime,
       ROUND(std_accel, 3) AS std_accel,
       ROUND(pct_rank,  3) AS pct_rank
FROM ranked
WHERE pct_rank >= 0.9
ORDER BY regime, pct_rank DESC;
"""
df_qE2 = pd.read_sql(qE2, conn)
display(df_qE2.head(15))

count_by_regime = df_qE2.groupby('regime').size().reset_index(name='high_uncertainty_flights')
display(count_by_regime)

fig, ax = plt.subplots(figsize=(6, 4))
colors = {'cruise': '#1976D2', 'landing': '#388E3C', 'takeoff': '#E91E63'}
ax.bar(count_by_regime['regime'], count_by_regime['high_uncertainty_flights'],
       color=[colors[r] for r in count_by_regime['regime']])
for i, (_, row) in enumerate(count_by_regime.iterrows()):
    ax.text(i, row['high_uncertainty_flights'] + 0.3, str(int(row['high_uncertainty_flights'])),
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Flights (top-10% variability)')
ax.set_title('RQ-E: High-Uncertainty Flights per Regime\n(PERCENT_RANK >= 0.9 on acceleration std dev)')
ax.set_ylim(0, 30)
plt.tight_layout()
out = os.path.join(_RESULTS_DIR, 'sql_qE2_uncertainty.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out}')

conn.close()
print('\nAll 5 SQL queries complete. Charts saved to visualization/')

,flight,regime,std_accel,pct_rank
0,141,cruise,1.033,1.000
1,85,cruise,0.907,0.995
2,77,cruise,0.878,0.989
3,139,cruise,0.875,0.984
4,116,cruise,0.835,0.979
5,138,cruise,0.732,0.973
6,80,cruise,0.703,0.968
7,86,cruise,0.701,0.963
8,79,cruise,0.689,0.957
9,83,cruise,0.676,0.952


,regime,high_uncertainty_flights
0,cruise,19
1,landing,19
2,takeoff,19


Saved: c:\Users\thien\code\AIMY\drone\sql\..\results\sql_qE2_uncertainty.png

All 5 SQL queries complete. Charts saved to visualization/
